# CinC 2019 LAMP Neural Audit

**Goal.** Audit neural and classical early-warning monitors trained on raw PhysioNet/CinC 2019 `.psv` trajectories, and show that higher AUC can be caused by evaluation gaming rather than valid early hidden-state inference.

- Source examples: 19508
- Held-out predictions: 6856
- Max patients: 8000
- Horizons: 6,12,18
- Early window: 12 hours

## Executive Result

LAMP distinguishes monitors with similar biomedical framing but different information contracts. Valid early-window models can pass the audit, while future-window and oracle-contaminated variants are flagged even when their AUC improves only modestly.

| Monitor                               | AUC   | Audit Status | Key Failures                          |
| ------------------------------------- | ----- | ------------ | ------------------------------------- |
| XGBoost valid                         | 0.841 | PASS         | none                                  |
| MLP hand-crafted valid                | 0.779 | PASS         | none                                  |
| TRANSFORMER hidden-state probe        | 0.672 | PASS         | none                                  |
| LSTM valid sequence                   | 0.661 | PASS         | none                                  |
| XGBoost future/oracle leaky           | 1.000 | FAIL         | temporal, forbidden, oracle proximity |
| MLP hand-crafted leaky                | 0.984 | FAIL         | temporal, forbidden, oracle proximity |
| TRANSFORMER future hidden-state probe | 0.706 | FAIL         | temporal, forbidden, oracle proximity |
| LSTM future-window leaky sequence     | 0.686 | FAIL         | temporal, forbidden, oracle proximity |

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import roc_curve, auc

ROOT = Path('..').resolve()
RESULTS = ROOT / 'results' / 'physionet_sequence_lamp'
summary = pd.read_csv(RESULTS / 'physionet_sequence_lamp_summary.csv')
metrics = pd.read_csv(RESULTS / 'physionet_sequence_metrics.csv')
registry = pd.read_csv(RESULTS / 'model_registry.csv')
predictions = pd.read_csv(RESULTS / 'physionet_sequence_predictions.csv')
summary.merge(registry, on='score', how='left').head()

## Model Comparison

The key table for the paper is not just `Model -> AUC`; it is `Model -> AUC -> Audit Status -> Failure Mode`.

| Model                                    | Type               | AUC   | Audit Status | Key Failures                          |
| ---------------------------------------- | ------------------ | ----- | ------------ | ------------------------------------- |
| LSTM future-window leaky sequence        | leaky              | 0.686 | FAIL         | temporal, forbidden, oracle proximity |
| LSTM valid sequence                      | valid              | 0.661 | PASS         | none                                  |
| MLP hand-crafted leaky                   | leaky              | 0.984 | FAIL         | temporal, forbidden, oracle proximity |
| MLP hand-crafted valid                   | valid              | 0.779 | PASS         | none                                  |
| Random Forest future/oracle leaky        | leaky              | 0.999 | FAIL         | temporal, forbidden, oracle proximity |
| Random Forest valid                      | valid              | 0.819 | PASS         | none                                  |
| TRANSFORMER future-window leaky sequence | leaky              | 0.692 | FAIL         | temporal, forbidden, oracle proximity |
| TRANSFORMER valid sequence               | valid              | 0.649 | PASS         | none                                  |
| TRANSFORMER hidden-state probe           | hidden probe       | 0.672 | PASS         | none                                  |
| TRANSFORMER future hidden-state probe    | leaky hidden probe | 0.706 | FAIL         | temporal, forbidden, oracle proximity |
| XGBoost future/oracle leaky              | leaky              | 1.000 | FAIL         | temporal, forbidden, oracle proximity |
| XGBoost valid                            | valid              | 0.841 | PASS         | none                                  |

## ROC Curves

![ROC curves](figures/cinc2019_roc_curves.png)

## Leakage-Dose Ablation

Low-dose oracle contamination is intentionally subtle: the AUC lift can be small, but LAMP still marks the score as using forbidden/oracle-adjacent information once the configured leakage-proximity geometry shifts.

| Family        | Leakage Dose | AUC   | Audit Status | Key Failures                          |
| ------------- | ------------ | ----- | ------------ | ------------------------------------- |
| LSTM          | 1pct         | 0.663 | FAIL         | temporal, forbidden                   |
| LSTM          | 5pct         | 0.673 | FAIL         | temporal, forbidden, oracle proximity |
| LSTM          | 10pct        | 0.685 | FAIL         | temporal, forbidden, oracle proximity |
| Random Forest | 1pct         | 0.823 | FAIL         | temporal, forbidden, oracle proximity |
| Random Forest | 5pct         | 0.828 | FAIL         | temporal, forbidden, oracle proximity |
| Random Forest | 10pct        | 0.841 | FAIL         | temporal, forbidden, oracle proximity |
| TRANSFORMER   | 1pct         | 0.651 | FAIL         | temporal, forbidden                   |
| TRANSFORMER   | 5pct         | 0.661 | FAIL         | temporal, forbidden, oracle proximity |
| TRANSFORMER   | 10pct        | 0.674 | FAIL         | temporal, forbidden, oracle proximity |
| XGBoost       | 1pct         | 0.842 | FAIL         | temporal, forbidden                   |
| XGBoost       | 5pct         | 0.846 | FAIL         | temporal, forbidden, oracle proximity |
| XGBoost       | 10pct        | 0.851 | FAIL         | temporal, forbidden, oracle proximity |
| mlp_probe     | 1pct         | 0.780 | FAIL         | temporal, forbidden                   |
| mlp_probe     | 5pct         | 0.784 | FAIL         | temporal, forbidden, oracle proximity |
| mlp_probe     | 10pct        | 0.789 | FAIL         | temporal, forbidden, oracle proximity |

![Leakage dose](figures/cinc2019_leakage_dose_auc_lift.png)

## Architecture and Horizon Ablation

The same audit contract is applied across horizons (6/12/18h), classical models, neural sequence models, and internal-representation probes.

| Score                   | 6     | 12    | 18    |
| ----------------------- | ----- | ----- | ----- |
| gru_valid_score         | 0.673 | 0.634 | 0.628 |
| lstm_valid_score        | 0.691 | 0.634 | 0.666 |
| mlp_probe_valid_score   | 0.780 | 0.800 | 0.781 |
| rf_valid_score          | 0.803 | 0.822 | 0.833 |
| transformer_valid_score | 0.669 | 0.659 | 0.622 |
| xgb_valid_score         | 0.841 | 0.830 | 0.857 |

![Horizon architecture ablation](figures/cinc2019_horizon_architecture_ablation.png)

## LAMP Gate Map

Green does not mean high AUC; green means the score survived the configured temporal, forbidden-feature, negative-control, matched-state, sentinel, and threshold gates.

![Audit gates](figures/cinc2019_audit_gates.png)

## Internal-State Probes

This is the alignment-native part: LAMP can audit not only final task scores but also probe outputs trained on model representations. Future-window hidden probes fail for the same reason a future-reading task model fails: the information contract has been broken.

| Probe                                 | AUC   | Audit Status | Key Failures                          |
| ------------------------------------- | ----- | ------------ | ------------------------------------- |
| TRANSFORMER hidden-state probe        | 0.672 | PASS         | none                                  |
| LSTM hidden-state probe               | 0.649 | PASS         | none                                  |
| TRANSFORMER future hidden-state probe | 0.706 | FAIL         | temporal, forbidden, oracle proximity |
| LSTM future hidden-state probe        | 0.680 | FAIL         | temporal, forbidden, oracle proximity |

## Timeline: Where Leakage Enters

The audit does not wait for clinical deployment to discover that a model has read forbidden information. The temporal contract is violated at the moment a score uses post-anchor physiology or label/onset-derived features.

![Timeline schema](figures/cinc2019_leakage_timeline_schema.png)

![Empirical timeline](figures/cinc2019_empirical_leakage_timeline.png)

## Interpretation for Evaluation Gaming

A neural early-warning system can look better under ordinary validation because it quietly learns from future physiology, oracle-like event proximity, or contaminated internal traces. LAMP reframes the result: performance is not validity. A model that passes LAMP has not proven clinical safety or alignment safety, but it has earned the right to more expensive prospective evaluation.

For AI alignment, the direct analogy is a latent monitor for deception or situational awareness. If the monitor wins by reading evaluation artifacts, post-hoc labels, privileged scratchpads, or hidden routing signals, it can obtain impressive AUC while failing the audit.